# ECON 8876 SP 24 Session 11 (Lab)
## Instructions
Please do the following for this lab. Don't forget to upload your .ipynb file to CarmenCanvas Assignments.

This lab compares NPL with NFXP and CCP


# Preliminaries

The NFXP code we used in the last lab is now written as .py files and we will call these later. Change the path to the location of the .py files and bus data on your google drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import time
import pandas as pd
import scipy.optimize as optimize
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

path = '/content/drive/MyDrive/Session_11/'
os.chdir(path)
!pwd

from model_zucher import zurcher
from Solve_NFXP import solve_NFXP
from estimate_NFXP import estimate as NFXP_estimate



# Policy function operator

The following code solves the model using successive approximations in CCP space. Lines 14-15 capture how we update the CCPs at each iteration:
$$
p^{k} = \Psi(p^{k-1}) = \Lambda(\varphi(p^{k-1}))
$$





In [ ]:
def NPLsolve():
    '''Solve model by successive approximation in choice probabilitiy space'''
    #Set starting valiues
    pk0 = np.ones((model.n))*0.99

    #Allocate
    pk = np.nan+np.zeros((100,model.n))
    pk[0,:] = pk0

    #Update grids
    model.create_grid()

    #Solve using successive approximation
    for i in range(1,100):
        pk[i,:]  = Psi(pk[i-1,:])

    return pk

def Psi(pk0,Finv=None):
    '''Evaluate Psi function (mapping from initial CCP to updated CCP)'''
    # Evaluate varphi function (find Vsigma implied by initial CCP)
    Vsigma = varphi(pk0, Finv)

    # Evaluate lambda function (find updated CCP implied by Vsigma)
    pk = lambdaa(Vsigma)

    return pk

def varphi(pk,Finv=None):
    '''Evaluate varphi function (mapping from initial CCP to Vsigma)'''
    #Calculate [I-Fu]^{-1} if not given

    if Finv is None:
        Finv = unc_state_transition(pk)

    # Euler constant
    eulerc = np.euler_gamma
    # Compute Vsigma
    value_keep = -model.cost + eulerc-np.log(pk)
    value_replace = -model.RC - model.cost[0] + eulerc - np.log(1-pk)
    pv = value_keep*pk + value_replace*(1-pk)
    Vsigma = np.ravel(Finv@pv)

    return Vsigma

def unc_state_transition(pk):
    #Calculate unconditional transition matrix
    Fu = model.P1*pk[:,np.newaxis] + model.P1[0,:]*(1-pk[:,np.newaxis])

    #Calculate [I-Fu]^{-1}
    Finv = np.linalg.inv(np.identity(model.n)-model.beta*Fu)

    return Finv

def lambdaa(Vsigma):
    '''Evaluate lambda function (mapping from Vsigma to updated CCP)'''
    #Compute choice probability
    value_keep = -model.cost + model.beta*model.P1@Vsigma
    value_replace = -model.RC - model.cost[0]  + model.beta*model.P1[0,:]@Vsigma
    pk=1/(1+np.exp(value_replace-value_keep))

    return  pk



Let's evaluate the PF operator and plot $p^k$ for $k=1,2,...,10$ and $k=99$ (the last iteration set in the code).

In [ ]:
# SETUP
model = zurcher()

# SOLVE THE MODEL
pk_NPL = NPLsolve()

fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(1,1,1)

ax.set_title(f'Convergence of $\Psi$')
ax.set_xlabel('Mileage grid')
ax.set_ylabel('Replacement probability')


Compare $p^k$ using NPL with $k=1,4,5$ with $p^k$ using NFXP.

In [ ]:
# solve the model with NFXP
ev0 = np.zeros(model.n)
solver_NFXP = solve_NFXP()
_, pk_NFXP = solver_NFXP.poly(model.bellman, ev0, beta=model.beta, output=2)

# Define figure
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(1,1,1)


ax.set_xlabel('Mileage grid')
ax.set_ylabel('Replacement probability')
plt.legend()

plt.show()

# Estimation

The code below sets up and estimates the model using NPL.

In [ ]:
def NPLsetup_data(data):

    # setup
    class data_class: pass
    data_class.x = data.x -1
    data_class.dx1= data.dx1
    data_class.dk = (data.d == 0)
    data_class.dr = (data.d == 1)
    return data_class

def NPLestimate(model, data, Kmax = 100):
    '''Estimate model using Nested Psuedo Likelihood (NPL)'''
    #Load previous choice probability as global variable
    global pk

    # Find transition probabilities, p (non-parametrical 1-step estimator)
    tabulate = data.dx1.value_counts() # Count number of observations for each dx1
    p = [tabulate[i]/sum(tabulate) if i < len(tabulate) else 0 for i in range(len(model.p))]
    model.p[:] = p # Use first step estimates as starting values for p


    # Find state transition matrix
    model.state_transition()

    #Set starting valiues
    pk0 = np.ones((model.n))*0.99  # starting value for CCP's
    theta0 = [0,0] # starting value for parameters

    #Outer loop: Will update CCPs until convergence or Kmax iterations are reached
    for _ in range(Kmax): #

        # Step 0)  Pre-compute unconditional transition matrix Fu and the inverse of I-beta*Fu to used in step 1
        Finv = unc_state_transition(pk0) # Calculate Fu and Finv

        #Inner loop:
        # Step 1)  Maximize the pseudo-likelihood function given step K-1 CCPs
        res = optimize.minimize(ll,theta0,args =(model, data, pk0, Finv), method='Newton-CG', jac = grad, hess= hes, tol = 1e-6)
        theta_hat = res.x # save parameters
        NPL_metric = np.abs(theta0-theta_hat) #save distance between parameters

        #Outer loop step
        # Step 2)  Update CCPs using theta_npl from step 1)
        pk0 = pk
        theta0 = theta_hat
        if NPL_metric.all() < 1e-6: # check convergence
            return res, theta_hat,pk

    #print(f'The function did not converge after {K} iterations')

    return res, theta_hat, pk

def ll(theta, model, data, pk0,Finv,out=1):
    '''Log-likelihood function for NPL'''
    #Load previous choice probability as global variable
    global pk

    # update parameters
    model.RC = theta[0]
    model.c = theta[1]
    model.create_grid()

    # Update CCPs
    pk = Psi(pk0,Finv)

    #Map choice probabilities to data
    pKdata = pk[data.x]

    #Return CCPs if out = 2
    if out == 2:
        return pk, pKdata

    #Calculate log-likelihood
    log_lik = np.log(data.dk*pKdata+(1-pKdata)*data.dr)

    #Return log-likelihood if out = 1
    f = -np.mean(log_lik)
    return f

def score(theta, model, data, pk0, Finv):
    global pk
    pk, pkdata = ll(theta, model, data, pk0, Finv, out=2)
    res = data.dk-pkdata

    NT = ((pkdata.size))
    score = np.zeros((NT,theta.size))
    dP = model.P1[0,:]-model.P1

    dvdRC = np.ravel(-1+model.beta*dP@Finv@(1-pk[:,np.newaxis])*(-1))
    dvdc = np.ravel(model.dc + model.beta*dP@Finv@(pk*(-model.dc)))
    score[:,0] = res*dvdRC[data.x]
    score[:,1] = res*dvdc[data.x]

    return score

def grad(theta, model, data, pk0, Finv):
    s = score(theta, model, data, pk0, Finv)
    return np.mean(s,0)

def hes(theta, model, data, pk0, Finv):
    s = score(theta, model, data, pk0, Finv)
    return s.T@s/data.x.shape[0]

Estimate the parameters of the model via CCP, NPL and NFXP. Play around with Kmax (line 3) which sets the number of outer loops in NPL (i.e. $k$-stage policy iteration estimator)

In [ ]:
# ESTIMATE
# setup
Kmax=  # Max number of outer loop iterations for NPL
pnames = ['RC','c']
model = zurcher()

# Load data
data = model.read_busdata(bustypes=[1,2,3,4])
data_class = NPLsetup_data(data)
samplesize = data_class.x.size

#Estimate using NPL 1-stage (i.e. CCP estimator)
import time
t0 = time.time()
res, theta_hat,pk = NPLestimate(model, data_class, 1)
t1 = time.time()

# Print the results
print(f'Structual estimation using busdata from Rust(1987)')
print(f'Beta        = {model.beta:.4f}')
print(f'n           = {model.n}')
print(f'Sample size = {samplesize}')

print(f'\n \nNPL 1-stage (CCP) \n--------------------------------')
print(f'Parameters     Estimates    ')
print(f'{pnames[0]}             {theta_hat[0]:.4f}     ')
print(f'{pnames[1]}              {theta_hat[1]:.4f}      \n ')


print(f'Log-likelihood {-res.fun*samplesize:.4f}')
print(f'runtime (seconds) {(t1-t0):.4f}')

#Estimate using NPL Kmax-stage (i.e. NPL-k)
import time
t0 = time.time()
res, theta_hat,pk = NPLestimate(model, data_class, Kmax)
t1 = time.time()

# Print the results
print(f'\n \nNPL {Kmax}-stage \n--------------------------------')
print(f'Parameters     Estimates   ')
print(f'{pnames[0]}             {theta_hat[0]:.4f}     ')
print(f'{pnames[1]}              {theta_hat[1]:.4f}      \n ')


print(f'Log-likelihood {-res.fun*samplesize:.4f}')
print(f'runtime (seconds) {(t1-t0):.4f}')

# Estimate using NFXP
theta0 = (0.,0.)
solver_NFXP = solve_NFXP()
t0 = time.time()
nfxp_model, optim_res, pnames, theta_hat, Avar, converged=NFXP_estimate(model, solver_NFXP,data,theta0=theta0, twostep=0)

t1 = time.time()
time = t1-t0

# Print the result
print('\n \nNFXP \n--------------------------------')
print(f'Parameters     Estimates   ')
print(f'{pnames[0]}             {theta_hat[0]:.4f}     ')
print(f'{pnames[1]}              {theta_hat[1]:.4f}     \n ')

print(f'Log-likelihood {-optim_res.fun*samplesize:.2f}')
print(f'runtime (seconds) {time:.4f}')